In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os, re, json, time, warnings
import numpy as np
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from langchain_openai import OpenAIEmbeddings,OpenAI
import getpass
from langchain_community.vectorstores import FAISS
from langchain_classic.schema import Document
from langchain_core.prompts import PromptTemplate

warnings.filterwarnings('ignore')

from pypdf import PdfReader

C:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdf(path: str) -> Tuple[List[str], Dict]:
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ""
        pages.append(text)

    meta = {
        "num_pages" : len(pages),
        "total_chars" : sum(len(p) for p in pages),
        "avg_chars_per_page": int(np.mean([len(p) for p in pages])),
        "empty_pages" : sum(1 for p in pages if len(p.strip()) < 50),
        "title" : reader.metadata.get("/Title", "Unknown"),
        "author" : reader.metadata.get("/Author", "Unknown"),
    }
    return pages, meta


def analyze_text_quality(pages: List[str]) -> Dict:
    issues = {
        "hyphenated_words": 0,
        "repeated_spaces"  : 0,
        "page_numbers_only": 0,
        "header_footer_noise": 0,
    }
    samples = []
    for p in pages:
        issues["hyphenated_words"] += len(re.findall(r'\w+-\n\w+', p))
        issues["repeated_spaces"] += len(re.findall(r' {3,}', p))
        issues["page_numbers_only"] += 1 if re.fullmatch(r'\s*\d+\s*', p) else 0
        if p.strip():
            samples.append(p[:120].replace('\n', ' '))
    return issues, samples[:3]


def clean_text(text: str) -> str:
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'^\s*\d+\s*$', '', text, flags=re.M)
    return text.strip()


In [3]:
pages, doc_meta = load_pdf('./intro-to-ml.pdf')
print(f"Loaded PDF: {doc_meta['num_pages']} pages")

quality_issues, sample_texts = analyze_text_quality(pages)
print("Text Quality Issues Detected:")
for issue, count in quality_issues.items():
    print(f"{issue:<28}: {count}")

cleaned_pages = [clean_text(p) for p in pages]
full_text = "\n\n".join(p for p in cleaned_pages if len(p) > 50)

Loaded PDF: 392 pages
Text Quality Issues Detected:
hyphenated_words            : 55
repeated_spaces             : 1572
page_numbers_only           : 0
header_footer_noise         : 0


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = text_splitter.split_text(full_text)
docs = [Document(page_content=chunk) for chunk in chunks]

print(f"\nSample Chunk [0]:")
print(docs[0].page_content[:400])


Sample Chunk [0]:
Andreas C. Müller & Sarah Guido
Introduction to 
Machine 
Learning 
with P y t h o n 
A GUIDE FOR DATA SCIENTISTS

Andreas C. Müller and Sarah Guido
Introduction to Machine Learning
with Python
A Guide for Data Scientists
Boston Farnham Sebastopol TokyoBeijing Boston Farnham Sebastopol TokyoBeijing


In [5]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key:  ········


In [6]:
embeddings = OpenAIEmbeddings()

In [7]:
vectorstore = FAISS.from_documents(docs, embeddings)

query = "What is backpropagation?"

results = vectorstore.similarity_search(query, k=5)

for k in [3, 5, 7]:
    print(f"\n--- k={k} ---")
    results = vectorstore.similarity_search_with_score(query, k=k)
    for i, (doc, score) in enumerate(results):
        print(f"Rank {i+1} | Score: {score:.4f}")
        print(doc.page_content[:200])
        print()


--- k=3 ---
Rank 1 | Score: 0.4128
learning applications, deep learning algorithms are often tailored very carefully to a
specific use case. Here, we will only discuss some relatively simple methods, namely
multilayer perceptrons for c

Rank 2 | Score: 0.4418
2 A preprint of Deep Learning can be viewed at http://www.deeplearningbook.org/.
already know about how the real world works. If the compass and accelerometer tell
you a user is going north, and the G

Rank 3 | Score: 0.4434
Figure 2-45. Illustration of a multilayer perceptron with a single hidden layer
This model has a lot more coefficients (also called weights) to learn: there is one
between every input and every hidden


--- k=5 ---
Rank 1 | Score: 0.4128
learning applications, deep learning algorithms are often tailored very carefully to a
specific use case. Here, we will only discuss some relatively simple methods, namely
multilayer perceptrons for c

Rank 2 | Score: 0.4418
2 A preprint of Deep Learning can be viewed at http

In [8]:
def retrieve(query: str, k: int = 5):
    results = vectorstore.similarity_search_with_score(query, k=k)
    return results

sample_query = "What is a decision tree?"

for k in [3, 5, 7]:
    print(f"\n{'='*50}")
    print(f" k = {k}")
    print(f"{'='*50}")
    results = retrieve(sample_query, k=k)
    for i, (doc, score) in enumerate(results):
        print(f"\nRank {i+1} | Score: {score:.4f}")
        print(doc.page_content[:300])
        print("-" * 40)


 k = 3

Rank 1 | Score: 0.2682
Decision trees are widely used models for classification and regression tasks. Essen‐
tially, they learn a hierarchy of if/else questions, leading to a decision.
These questions are similar to the questions you might ask in a game of 20 Questions.
Imagine you want to distinguish between the followin
----------------------------------------

Rank 2 | Score: 0.2965
ted decision trees.
Random forests
As we just observed, a main drawback of decision trees is that they tend to overfit the
training data. Random forests are one way to address this problem. A random forest
is essentially a collection of decision trees, where each tree is slightly different from
the 
----------------------------------------

Rank 3 | Score: 0.3022
The main downside of decision trees is that even with the use of pre-pruning, they
tend to overfit and provide poor generalization performance. Therefore, in most
applications, the ensemble methods we discuss next are usually used in pl

In [9]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
You are a Machine Learning teaching assistant.
Answer the question using ONLY the context provided below.
If the answer is not found in the context, say:
"I could not find relevant information in the provided context."
Do NOT use any external knowledge or make assumptions.

Context:
{context}

Question: {question}

Answer:"""
)

llm = OpenAI()


def generate_answer(query: str, k: int = 5) -> str:
    results = retrieve(query, k=k)
    context = "\n\n---\n\n".join([doc.page_content for doc, score in results])
    prompt = prompt_template.format(context=context, question=query)
    response = llm.invoke(prompt)
    return response.content if hasattr(response, 'content') else response


sample_queries = [
    "What is a decision tree and how does it work?",
    "How does k-nearest neighbors classify data?",
    "What is overfitting and how can it be prevented?",
]

for query in sample_queries:
    print(f"\n{'='*60}")
    print(f"Q: {query}")
    print(f"{'='*60}")
    answer = generate_answer(query, k=5)
    print(f"A: {answer}")


Q: What is a decision tree and how does it work?
A: 
A decision tree is a widely used model in machine learning for classification and regression tasks. It works by learning a hierarchy of if/else questions, similar to the questions asked in a game of 20 questions. The tree consists of nodes representing questions and terminal nodes containing the answer. The edges connect the answers to a question with the next question to ask. The tree is built by recursively partitioning the data based on the features until each leaf only contains a single target value. This process can be repeated to build multiple decision trees, and random forests use this strategy by injecting randomness into the tree building process to ensure each tree is different. 

Q: How does k-nearest neighbors classify data?
A:  k-nearest neighbors classifies data by finding the closest data points in the training dataset and assigning a label based on the majority class among these nearest neighbors. The value of k, or

In [11]:
def rag_pipeline(query: str, k: int = 5, verbose: bool = True) -> dict:
    results = retrieve(query, k=k)
    context_chunks = [doc.page_content for doc, score in results]
    scores = [round(score, 4) for doc, score in results]
    context = "\n\n---\n\n".join(context_chunks)

    prompt = prompt_template.format(context=context, question=query)
    answer = llm.invoke(prompt)

    if verbose:
        print(f"\n{'='*60}")
        print(f" QUERY: {query}")
        print(f"{'='*60}")
        print(f"\n Retrieved {k} chunks | Scores: {scores}")
        print(f"\n ANSWER:\n{answer}")
        print(f"{'='*60}")

    return {"query": query, "answer": answer, "chunks": context_chunks, "scores": scores}


demo_queries = [
    "What is a decision tree and how does it work?",
    "Explain the difference between classification and regression.",
    "What is cross-validation and why is it important?",
    "How does a support vector machine find the decision boundary?",
    "What are the main types of machine learning?",
]

for query in demo_queries:
    rag_pipeline(query, k=5)


 QUERY: What is a decision tree and how does it work?

 Retrieved 5 chunks | Scores: [np.float32(0.282), np.float32(0.3201), np.float32(0.3234), np.float32(0.3256), np.float32(0.3286)]

 ANSWER:

A decision tree is a model used for classification and regression tasks. It learns a hierarchy of if/else questions to make decisions. This process is similar to playing 20 questions, where each question narrows down the possibilities until a final decision is reached. In machine learning, decision trees are built using data and supervised learning techniques. The model is built by recursively splitting the data into smaller subsets, with each split based on a single feature. This process continues until each subset only contains a single class or target value. To reduce overfitting, random forests are often used, which involve building multiple decision trees with slight variations in the data and features used for each split. 

 QUERY: Explain the difference between classification and regre

In [ ]:
print("ML Knowledge Assistant — type 'quit' to exit\n")

while True:
    query = input("Your question: ").strip()
    if query.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not query:
        continue
    rag_pipeline(query, k=5)

ML Knowledge Assistant — type 'quit' to exit



Your question:  Explain about window transformers



 QUERY: Explain about window transformers

 Retrieved 5 chunks | Scores: [np.float32(0.4711), np.float32(0.5145), np.float32(0.5156), np.float32(0.5204), np.float32(0.5236)]

 ANSWER:

I could not find relevant information in the provided context.


Your question:  how is contamination rate calculated in isolation forest



 QUERY: how is contamination rate calculated in isolation forest

 Retrieved 5 chunks | Scores: [np.float32(0.5625), np.float32(0.5762), np.float32(0.5812), np.float32(0.583), np.float32(0.5841)]

 ANSWER:

The contamination rate in isolation forest is calculated as the proportion of outliers in the dataset. It is determined by dividing the number of outliers by the total number of observations in the dataset.


Your question:  what steps do i need to take if i need to build a model from scratch ?



 QUERY: what steps do i need to take if i need to build a model from scratch ?

 Retrieved 5 chunks | Scores: [np.float32(0.3971), np.float32(0.4049), np.float32(0.4102), np.float32(0.4658), np.float32(0.4661)]

 ANSWER:
 The next steps are usually acquiring the data and building a working prototype. In this book we have talked about many models you can employ, and how to properly evaluate and tune these models. While trying out models, though, keep in mind that this is only a small part of a larger data science workflow, and model building is often part of a feedback circle of collecting new data, cleaning data, building the model, evaluating its performance, and iterating on the process.
